In [1]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain.tools import Tool

# 1. Define a "Search PDF" Tool
def search_university_docs(query):
    # This uses the retriever logic you just perfected
    docs = retriever.invoke(query)
    return "\n\n".join([d.page_content for d in docs])

university_tool = Tool(
    name="KazNU_Knowledge_Base",
    func=search_university_docs,
    description="Use this tool to answer questions about KazNU rules, policies, and departments."
)

# 2. Add a "Web Search" Tool (For current info not in PDFs)
from langchain_community.tools import DuckDuckGoSearchRun
web_tool = DuckDuckGoSearchRun()

tools = [university_tool, web_tool]

# 3. Create the Agent (This is where the "Brain" begins)
prompt = hub.pull("hwchase17/react") # Standard Reasoning prompt
# ... (rest of the agent logic)

d:\RAG_StudentSupport_Project\agentic-rag-edu-4th\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (d:\RAG_StudentSupport_Project\agentic-rag-edu-4th\.venv\Lib\site-packages\langchain\agents\__init__.py)

In [2]:
import os
from dotenv import load_dotenv

# Core Agent Imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

# 1. SETUP & DB LOADING
load_dotenv()
DB_PATH = "../db"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 2. DEFINE THE TOOLS
# Tool A: The PDF Search (Your previous RAG logic)
def search_kaznu_policy(query: str):
    docs = retriever.invoke(query)
    return "\n\n".join([d.page_content for d in docs])

kaznu_tool = Tool(
    name="KazNU_Policy_Search",
    func=search_kaznu_policy,
    description="Use this for any questions regarding KazNU internal rules, academic policy, mobility requirements, and Keremet center."
)

# Tool B: Web Search (For real-time info)
web_search_tool = DuckDuckGoSearchRun()

tools = [kaznu_tool, web_search_tool]

# 3. INITIALIZE THE BRAIN (The LLM)
llm = ChatGoogleGenerativeAI(
    model="models/gemini-1.5-flash",
    temperature=0, # Agents need 0 temperature for stability
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# 4. PULL THE REASONING PROMPT
# This prompt tells the AI how to use Thought, Action, Action Input, and Observation
prompt = hub.pull("hwchase17/react")

# 5. CONSTRUCT THE AGENT
agent = create_react_agent(llm, tools, prompt)

# AgentExecutor is the runtime that handles the loop
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, # VERY IMPORTANT: This shows your supervisor the AI's "thinking"
    handle_parsing_errors=True,
    max_iterations=5
)

# 6. TEST THE AGENTIC REASONING
# This question requires BOTH tools to answer fully.
complex_query = (
    "What are the KazNU requirements for international mobility, "
    "and what are 3 famous universities in Poland I could potentially visit?"
)

print(f"--- AGENT STARTING TASK ---")
response = agent_executor.invoke({"input": complex_query})

print("\n--- FINAL AGENT RESPONSE ---")
print(response["output"])

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (d:\RAG_StudentSupport_Project\agentic-rag-edu-4th\.venv\Lib\site-packages\langchain\agents\__init__.py)